# Step 10 — Teammate features in the final model

Two teammate branches proposed new signals but stopped before testing them in the ranker:

| Branch | Proposal | What was missing |
|---|---|---|
| `feature/add-new-features-rh` (Step 4e) | **Purchase-sequence affinity**: is this candidate in a level-2 category that people tend to buy within a week after buying a category this user bought in the last 30 days? Timed purchase episodes, smoothed lift, recency decay, training-only fitting with 3-fold cross-fitting | Built on a 20,000-record sample only; no ranking evaluation |
| `weiming/feature-validation` (Step 8) | **Click context**: clicks on this exact candidate split by source (search vs. non-search) and recency (1 day vs. 7 days) | Already tested in Step 8: no gain (validation NDCG@10 −0.0002, 95% CI −0.0009 to +0.0004; Purchase-HR@10 −0.0043, CI excludes 0) |

Because Step 8 settled the click-context question, those four columns are built and described here (Section 2) but **not** retrained. This notebook builds the purchase-sequence features on **all 173,831 records** with the branch's own definition, adds them to the Step 6 LightGBM, decides on validation, and only then reports test results with a paired bootstrap. The winning LightGBM then replaces the old one inside the Step 6 ensemble (same z-score blend, same 0.5 weight chosen on validation in Step 6), so the final model can be compared directly with NDCG@10 0.4201.

**Inputs:** `features.parquet` (Steps 2–4b), `events.parquet`, `users.parquet`, `product_meta_data.txt` (category column only) and `step6_test_predictions.parquet` (neural test scores).
**Outputs (new files only; `features.parquet` is not modified):** `step10_new_features.parquet`, `step10_results.csv`, `step10_test_predictions.parquet`.
**Runtime:** about 30 minutes on a Colab **High-RAM CPU** runtime (no GPU needed).

A data fact checked while preparing this step: the test query appears in the user's *own* history for only 1 of 173,831 records (71% of test queries were searched by someone else before). JDSearch evidently removes repeat searches, so "clicked this item under the same query" features are not possible.

In [1]:
import os, time, gc, math
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import pyarrow as pa, pyarrow.csv as pacsv, pyarrow.parquet as pq, pyarrow.dataset as ds
import warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'
SFX = '_smoke' if SMOKE else ''
SEED = 42
PRED_PATH = Path(os.environ.get('JD_PRED', OUT_DIR / f'step6_test_predictions.parquet'))
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']
CLICK, ORD = 0, 2                                   # Step 1 event type codes
nU = len(pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user']))
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.03 if SMOKE else np.ones(nU, bool)
print(f'{in_scope.sum():,} records in scope' + (' | SMOKE TEST' if SMOKE else ''))

Mounted at /content/drive
173,831 records in scope


## 1. Load the feature table and history

Rows of `features.parquet` follow the candidate order of Step 1; the new columns are built in that same order and checked against it before anything is joined.

In [2]:
BASE = [c for c in pq.read_schema(OUT_DIR / 'features.parquet').names
        if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
flt = ds.field('user').isin(pa.array(np.flatnonzero(in_scope).astype(np.int32))) if SMOKE else None
feat = ds.dataset(OUT_DIR / 'features.parquet').to_table(columns=['user', 'wid', 'label', 'split'] + BASE,
                                                         filter=flt).to_pandas()
ev = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days', 'qry'])
ev = ev[in_scope[ev['user'].to_numpy()]].reset_index(drop=True)
assert (np.diff(ev['user'].to_numpy()) >= 0).all(), 'events must be sorted by user, then history position'
assert not feat.duplicated(['user', 'wid']).any()
print(f'baseline features: {len(BASE)} | candidate rows {len(feat):,} | history events {len(ev):,}')

def key(user, other):                    # pack (user, id) into one int64, as in Step 2
    return (np.asarray(user, np.int64) << 27) | (np.asarray(other, np.int64) & ((1 << 27) - 1))
CK, ORIG_WID = key(feat['user'], feat['wid']), feat['wid'].to_numpy()
NEW = {}

baseline features: 55 | candidate rows 15,510,012 | history events 26,667,260


## 2. Click context (from `weiming/feature-validation`)

The existing `uc_click_7d` counts all clicks the user made on this exact candidate in the last 7 days. The branch splits that count two ways:

- **Source.** A history event with query code `-1` happened outside search (homepage, recommendations, cart page); any other code was search-linked. 72% of recorded clicks are non-search.
- **Recency.** Last 24 hours vs. the full 7-day click window.

| Column | Meaning |
|---|---|
| `uc_click_search_1d`, `uc_click_search_7d` | Clicks on this candidate made from a search results page, last 1 / 7 days |
| `uc_click_queryless_1d`, `uc_click_queryless_7d` | Clicks on this candidate made outside search, last 1 / 7 days |

The two 7-day columns sum to `uc_click_7d`, so the new information is the split, not the count. Only history events enter; candidate labels are never read.

In [3]:
ev_click, ev_age, ev_qry = ev['type'].to_numpy() == CLICK, ev['age_days'].to_numpy(), ev['qry'].to_numpy()
EK = key(ev['user'], ev['wid'])

def count_on_candidates(mask):
    s = pd.Series(EK[mask]).value_counts()
    if not len(s):
        return np.zeros(len(CK), np.float32)
    i = s.index.get_indexer(CK)
    return np.where(i >= 0, s.to_numpy()[np.maximum(i, 0)], 0).astype(np.float32)

for days in [1, 7]:
    NEW[f'uc_click_search_{days}d'] = count_on_candidates(ev_click & (ev_age < days) & (ev_qry >= 0))
    NEW[f'uc_click_queryless_{days}d'] = count_on_candidates(ev_click & (ev_age < days) & (ev_qry == -1))
assert np.allclose(NEW['uc_click_search_7d'] + NEW['uc_click_queryless_7d'], feat['uc_click_7d'].to_numpy())
CLICK_FEATS = [c for c in NEW]
del EK; gc.collect()
print('click context built:', CLICK_FEATS)

click context built: ['uc_click_search_1d', 'uc_click_queryless_1d', 'uc_click_search_7d', 'uc_click_queryless_7d']


## 3. Purchase-sequence affinity (from `feature/add-new-features-rh`, Step 4e)

The branch's definition, unchanged:

1. **Purchase episodes.** Per user and level-2 category, orders within 30 minutes of the first order form one episode (a basket, not separate decisions).
2. **Sequence pairs.** For each episode in category A at least 8 days old, record every *other* category B bought 1 hour – 7 days later (counted once per episode). Episodes with no follow-up still count in the denominator.
3. **Smoothed lift.** `P(B | A) = (n_AB + 20·p_B) / (n_A + 20)`, `lift = P(B | A) / p_B`. Kept only if A has ≥ 50 episodes, the pair is seen in ≥ 10 distinct records, and lift > 1.
4. **Candidate score.** For each category A the user bought in the last 30 days, score `exp(−age/7) · log(lift(A→B))` for the candidate's category B; keep the best.

**Leakage control (the branch's design).** Associations are fitted on *training* records only. Each training record is scored with a table fitted on the other two thirds of the training records (3-fold cross-fitting), so a record never scores itself; validation and test use the table fitted on all training records.

The functions in the next cell are copied verbatim from the branch notebook. The only change is scale: the branch scored candidates one at a time in Python on a 20,000-record sample; here the scoring step is vectorised for 15.5 M candidates and checked against the branch's own `transform()` on 300 records.

| Column | Meaning | No relationship |
|---|---|---|
| `seq_score` | Best recency-decayed log-lift from a recent purchase to this candidate's category | 0 |
| `seq_support_log` | log(1 + distinct training records supporting that relationship) | 0 |
| `seq_source_age_days` | Age of the purchase that produced it | missing |

The branch's fourth column, `sequence_available`, equals `seq_score > 0`, so trees get it for free and it is dropped.

In [4]:
# --- verbatim from step4e_purchase_sequence_affinity.ipynb (branch feature/add-new-features-rh) ---
@dataclass(frozen=True)
class Config:
    episode_minutes: float = 30
    min_delay_hours: float = 1
    max_delay_days: float = 7
    buffer_days: float = 1
    alpha: float = 20
    min_source_episodes: int = 50
    min_pair_records: int = 10
    recent_days: float = 30
    decay_days: float = 7

    def __post_init__(self):
        if not all(math.isfinite(v) for v in asdict(self).values()):
            raise ValueError("Configuration must be finite")
        if (self.episode_minutes < 0 or self.min_delay_hours < 0
                or self.max_delay_days < self.min_delay_hours / 24
                or self.buffer_days < 0 or self.alpha <= 0
                or self.min_source_episodes < 1 or self.min_pair_records < 1
                or self.recent_days < 0 or self.decay_days <= 0):
            raise ValueError("Invalid feature configuration")


def episodes(orders, config=Config()):
    """orders: chronological (category, age_days); missing categories excluded.

    A fixed per-category anchor starts each episode; its representative is the
    last included event. Stable event index resolves equal representative times.
    """
    active, result = {}, []
    previous = math.inf
    for index, (category, age) in enumerate(orders):
        if not math.isfinite(age) or age < 0 or age > previous:
            raise ValueError("Orders must have finite non-increasing ages")
        previous = age
        if category is None:
            continue
        if category in active:
            anchor, last_age, last_index = active[category]
            if anchor - age <= config.episode_minutes / 1440 + 1e-12:
                active[category] = (anchor, age, index)
                continue
            result.append((category, last_age, last_index))
        active[category] = (age, age, index)
    result.extend((cat, age, index) for cat, (_, age, index) in active.items())
    result.sort(key=lambda x: (-x[1], x[2]))
    return [(cat, age) for cat, age, _ in result]


def count_associations(histories, config=Config()):
    """Count source windows, including empty windows, in distinct records.

    histories is an iterable of chronological order lists, one per original
    record. No labels, query text, or candidate products are accepted here.
    """
    sources, pairs, destinations, records = Counter(), Counter(), Counter(), Counter()
    for orders in histories:
        ep = episodes(orders, config)
        record_pairs = set()
        for i, (source, age) in enumerate(ep):
            if age + 1e-12 < config.max_delay_days + config.buffer_days:
                continue
            sources[source] += 1
            seen = set()
            for j in range(i + 1, len(ep)):
                dest, dest_age = ep[j]
                delay = age - dest_age
                if delay > config.max_delay_days + 1e-12:
                    break
                if delay + 1e-12 >= config.min_delay_hours / 24 and dest != source:
                    seen.add(dest)
            for dest in seen:
                pairs[source, dest] += 1
                destinations[dest] += 1
                record_pairs.add((source, dest))
        records.update(record_pairs)
    return dict(sources=sources, pairs=pairs, destinations=destinations, records=records)


def association_rows(counts, config=Config()):
    """All observed pairs with explicit support and positive-lift flags."""
    total = sum(counts['sources'].values())
    for (source, dest), n_ab in sorted(counts['pairs'].items()):
        n_a, u_ab = counts['sources'][source], counts['records'][source, dest]
        p_b = counts['destinations'][dest] / total if total else 0
        if p_b <= 0:
            continue
        probability = (n_ab + config.alpha * p_b) / (n_a + config.alpha)
        lift = probability / p_b
        support = n_a >= config.min_source_episodes and u_ab >= config.min_pair_records
        yield dict(source_category=source, destination_category=dest,
                   source_episodes=n_a, pair_episodes=n_ab, pair_records=u_ab,
                   probability=probability, background_probability=p_b, lift=lift,
                   support_threshold_met=support, positive_lift=lift > 1,
                   supported=support and lift > 1)

def scoring_lookup(rows):
    return {(r['source_category'], r['destination_category']):
            (r['lift'], r['pair_records']) for r in rows if r['supported']}

def fit_table(histories, out_dir, name, config):
    counts = count_associations(histories, config)
    rows = list(association_rows(counts, config))
    frame = pd.DataFrame(rows, columns=ASSOCIATION_COLUMNS)
    save_parquet(frame, out_dir / f'associations_{name}.parquet')
    # Source denominators include categories that never produce a pair.
    sources = pd.DataFrame(sorted(counts['sources'].items()), columns=['source_category', 'source_episodes'])
    save_parquet(sources, out_dir / f'source_windows_{name}.parquet')
    diag = dict(eligible_windows=sum(counts['sources'].values()), observed_pairs=len(rows),
                supported_positive_pairs=sum(r['supported'] for r in rows))
    print(f'{name}: {diag}', flush=True)
    return scoring_lookup(rows), diag


def transform(orders, destination_categories, lookup, config=Config()):
    """One tuple per candidate occurrence; repeated categories reuse scores.

    Equal scores select the smallest integer source category. Missing category
    is None, never a shared artificial category. Availability distinguishes a
    missing age from a meaningful age of zero.
    """
    recent = {}
    for cat, age in orders:
        if cat is not None and 0 <= age <= config.recent_days:
            recent[cat] = min(recent.get(cat, math.inf), age)
    cache = {}
    for dest in destination_categories:
        if dest not in cache:
            selected = (0.0, 0.0, math.nan, 0)
            if dest is not None:
                for source in sorted(recent):
                    if source == dest or (source, dest) not in lookup:
                        continue
                    lift, support = lookup[source, dest]
                    if lift <= 1:
                        continue
                    age = recent[source]
                    score = math.exp(-age / config.decay_days) * math.log(lift)
                    if score > selected[0]:
                        selected = (score, math.log1p(support), age, 1)
            cache[dest] = selected
        yield cache[dest]


In [5]:
cfg = Config()
cols = ['wid', 'cate_id_2']
meta = pacsv.read_csv(DATA_DIR / 'product_meta_data.txt',
    parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
    convert_options=pacsv.ConvertOptions(include_columns=cols, column_types={c: pa.int64() for c in cols})).to_pandas()
meta_idx = pd.Index(meta['wid'])
def cat2(w):
    i = meta_idx.get_indexer(w)
    c = np.where(i >= 0, meta['cate_id_2'].to_numpy()[np.maximum(i, 0)], -1)
    return np.where(c >= 0, c, -1)            # unknown or negative category never matches anything

o = ev[ev['type'] == ORD]                     # already in (user, history position) order = chronological
o_user, o_age, o_cat = o['user'].to_numpy(), o['age_days'].to_numpy().astype(np.float64), cat2(o['wid'].to_numpy())
cand_cat = cat2(feat['wid'].to_numpy())
del meta, meta_idx, o, ev, ev_click, ev_age, ev_qry; gc.collect()
print(f'orders {len(o_user):,} | known category: orders {np.mean(o_cat >= 0):.1%}, candidates {np.mean(cand_cat >= 0):.1%}')

# per-record chronological order lists, the branch's input format
starts = np.r_[0, np.flatnonzero(o_user[1:] != o_user[:-1]) + 1] if len(o_user) else np.array([], int)
ends = np.r_[starts[1:], len(o_user)]
orders = {int(o_user[a]): [(None if c < 0 else int(c), float(x)) for c, x in zip(o_cat[a:b], o_age[a:b])]
          for a, b in zip(starts, ends)}

split_of = feat.groupby('user')['split'].first().reindex(range(nU)).to_numpy()
train_u = np.flatnonzero(split_of == 'train')
fold = np.full(nU, -1, np.int8)                                   # -1 = validation/test record
fold[train_u[np.random.default_rng(SEED + 1).permutation(len(train_u))]] = np.arange(len(train_u)) % 3

t0 = time.time(); TABLES = {}; FIT = {}
for f in [0, 1, 2, 'all_train']:
    hist = (orders.get(int(u), []) for u in train_u if f == 'all_train' or fold[u] != f)
    counts = count_associations(hist, cfg)
    rows = list(association_rows(counts, cfg))
    TABLES[f] = pd.DataFrame([(r['source_category'], r['destination_category'], r['lift'], r['pair_records'])
                              for r in rows if r['supported']], columns=['src', 'dst', 'lift', 'support'])
    FIT[f] = dict(eligible_episodes=sum(counts['sources'].values()), observed_pairs=len(rows),
                  supported_pairs=len(TABLES[f]))
print(f'associations fitted in {time.time() - t0:.0f}s')
pd.DataFrame(FIT).T

orders 6,817,283 | known category: orders 91.1%, candidates 99.5%
associations fitted in 52s


,eligible_episodes,observed_pairs,supported_pairs
0,2097162,93215,15636
1,2096040,94095,16120
2,2094508,93822,15960
all_train,3143855,103282,18876


In [6]:
# vectorised transform(): recent source categories x supported pairs -> best score per (record, destination)
rec = pd.DataFrame({'user': o_user, 'src': o_cat, 'age': o_age})
rec = rec[(rec['src'] >= 0) & rec['age'].between(0, cfg.recent_days)].groupby(['user', 'src'], as_index=False)['age'].min()
rec['table'] = fold[rec['user'].to_numpy()]
parts = []
for f, name in [(0, 0), (1, 1), (2, 2), (-1, 'all_train')]:
    j = rec[rec['table'] == f].merge(TABLES[name], on='src')
    parts.append(j[j['src'] != j['dst']])
j = pd.concat(parts, ignore_index=True)
j['score'] = np.exp(-j['age'] / cfg.decay_days) * np.log(j['lift'])
best = (j.sort_values(['user', 'dst', 'score', 'src'], ascending=[True, True, False, True])   # ties: smallest source
         .drop_duplicates(['user', 'dst']))
i = pd.Index(key(best['user'], best['dst'])).get_indexer(np.where(cand_cat >= 0, key(feat['user'], cand_cat), -1))
hit, bi = i >= 0, np.maximum(i, 0)
NEW['seq_score'] = np.where(hit, best['score'].to_numpy()[bi], 0).astype(np.float32)
NEW['seq_support_log'] = np.where(hit, np.log1p(best['support'].to_numpy()[bi]), 0).astype(np.float32)
NEW['seq_source_age_days'] = np.where(hit, best['age'].to_numpy()[bi], np.nan).astype(np.float32)
SEQ_FEATS = ['seq_score', 'seq_support_log', 'seq_source_age_days']

# equivalence check against the branch's own transform()
cu = feat['user'].to_numpy()
st = np.r_[0, np.flatnonzero(cu[1:] != cu[:-1]) + 1]; en = np.r_[st[1:], len(cu)]
slate_of = pd.Series(np.arange(len(st)), index=cu[st])
mismatch = 0
for u in np.random.default_rng(0).choice(rec['user'].unique(), min(300, rec['user'].nunique()), replace=False):
    a, b = st[slate_of[u]], en[slate_of[u]]
    tab = TABLES['all_train' if fold[u] < 0 else int(fold[u])]
    lookup = {(s, d): (l, n) for s, d, l, n in tab.itertuples(index=False)}
    ref = transform(orders.get(int(u), []), [None if c < 0 else int(c) for c in cand_cat[a:b]], lookup, cfg)
    for (s1, l1, _, _), s2, l2 in zip(ref, NEW['seq_score'][a:b], NEW['seq_support_log'][a:b]):
        mismatch += abs(s1 - s2) > 1e-5 or abs(l1 - l2) > 1e-5
assert mismatch == 0, f'{mismatch} candidates differ from the branch transform()'
print('vectorised scores match the branch transform() on 300 records')
del orders, rec, j, best, parts; gc.collect()

vectorised scores match the branch transform() on 300 records


9207

## 4. Coverage and raw association with the label

"Clicked rate" and "purchase rate" compare candidates where the feature is present with those where it is absent. This is a pooled comparison, so user-level effects (frequent buyers have both more recent orders and more purchases) are mixed in; the ranking test in Section 5 is the within-slate answer.

In [7]:
lab = feat['label'].to_numpy()
rows = {}
for c in CLICK_FEATS + SEQ_FEATS:
    on = ~np.isnan(NEW[c]) if c == 'seq_source_age_days' else NEW[c] > 0
    rows[c] = {'candidates with it': on.mean(), 'slates with it': pd.Series(on).groupby(cu).any().mean(),
               'interacted rate (present)': (lab[on] > 0).mean(), 'interacted rate (absent)': (lab[~on] > 0).mean(),
               'purchase rate (present)': (lab[on] == 3).mean(), 'purchase rate (absent)': (lab[~on] == 3).mean()}
    if c == 'seq_score':
        varies = pd.Series(NEW[c]).groupby(cu).nunique() > 1
        print(f'slates where seq_score differs between candidates (can change the ranking): {varies.mean():.1%}')
coverage = pd.DataFrame(rows).T
coverage.round(4)

slates where seq_score differs between candidates (can change the ranking): 46.2%


,candidates with it,slates with it,interacted rate (present),interacted rate (absent),purchase rate (present),purchase rate (absent)
uc_click_search_1d,0.0025,0.1147,0.3144,0.0200,0.0198,0.0013
uc_click_queryless_1d,0.0023,0.1143,0.2814,0.0201,0.0231,0.0013
uc_click_search_7d,0.0035,0.1448,0.2803,0.0198,0.0166,0.0013
uc_click_queryless_7d,0.0033,0.1438,0.2539,0.0199,0.0188,0.0013
seq_score,0.4563,0.5445,0.0210,0.0204,0.0018,0.0009
seq_support_log,0.4563,0.5445,0.0210,0.0204,0.0018,0.0009
seq_source_age_days,0.4563,0.5445,0.0210,0.0204,0.0018,0.0009


## 5. LightGBM: baseline vs. purchase sequence

Same configuration as the best Step 6 LightGBM (LambdaRank, `label_gain` [0, 1, 3, 7], learning rate 0.03, up to 1,200 trees, 63 leaves, early stopping on validation NDCG@10), the same shuffled slates and the same 55 baseline features (seasonal columns excluded). The two variants differ only in their feature list. **The variant is chosen on validation**; test is used once afterwards.

In [8]:
for c in CLICK_FEATS + SEQ_FEATS:
    feat[c] = NEW[c]
feat = feat.iloc[np.lexsort((np.random.default_rng(SEED).random(len(feat)), feat['user'].to_numpy()))]
feat = feat[feat.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)
part = {s: feat.index[feat['split'] == s].to_numpy() for s in ['train', 'val', 'test']}
groups = {s: feat.loc[part[s]].groupby('user', sort=False).size().to_numpy() for s in part}
print({s: f'{len(g):,} slates' for s, g in groups.items()})

TREES = 60 if SMOKE else 1200
LGB = dict(objective='lambdarank', n_estimators=TREES, learning_rate=0.03, num_leaves=63, min_child_samples=200,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1,
           label_gain=[0, 1, 3, 7])

def slate_metrics(u, lab, score, k=10, seed=0):
    lab = lab.astype(np.float64)
    order = np.lexsort((np.random.default_rng(seed).random(len(u)), -np.asarray(score, np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    gid = np.repeat(np.arange(len(start)), size)
    disc = np.where(rank < k, 1.0 / np.log2(rank + 2), 0.0)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    idcg = np.bincount(gid, weights=(2 ** lab[np.lexsort((-lab, u))] - 1) * disc)
    has_buy = np.bincount(gid, weights=lab_s == 3) > 0
    return {'user': u_s[start], 'ndcg': dcg / idcg, 'hit': np.bincount(gid, weights=(lab_s > 0) & (rank < k)) > 0,
            'has_buy': has_buy, 'buy_hit': np.bincount(gid, weights=(lab_s == 3) & (rank < k)) > 0}

def summary(m):
    return {'NDCG@10': m['ndcg'].mean(), 'HR@10': m['hit'].mean(), 'Purchase-HR@10': m['buy_hit'][m['has_buy']].mean()}

VARIANTS = {'baseline (Step 6 LightGBM)': BASE, '+ purchase sequence': BASE + SEQ_FEATS}   # click context: see Step 8
U = {s: feat['user'].to_numpy()[part[s]] for s in part}
Y = {s: feat['label'].to_numpy()[part[s]] for s in part}
grid, SCORES, IMP = {}, {}, {}
for name, cols in VARIANTS.items():
    t0 = time.time()
    X = {s: feat.loc[part[s], cols] for s in part}
    m = lgb.LGBMRanker(**LGB).fit(X['train'], Y['train'], group=groups['train'],
                                  eval_set=[(X['val'], Y['val'])], eval_group=[groups['val']], eval_at=[10],
                                  callbacks=[lgb.early_stopping(50, verbose=False)])
    SCORES[name] = {s: m.predict(X[s]) for s in ('val', 'test')}
    imp = pd.Series(m.booster_.feature_importance('gain'), index=cols); IMP[name] = imp / imp.sum()
    grid[name] = {**{f'val {k}': v for k, v in summary(slate_metrics(U['val'], Y['val'], SCORES[name]['val'])).items()},
                  'trees': m.best_iteration_, 'gain share of new features': IMP[name].drop(BASE).sum(),
                  'min': (time.time() - t0) / 60}
    print(f"{name}: val NDCG@10 {grid[name]['val NDCG@10']:.4f} | {grid[name]['min']:.1f} min", flush=True)
    del X; gc.collect()
GRID = pd.DataFrame(grid).T
GRID.round(4)

{'train': '120,081 slates', 'val': '25,825 slates', 'test': '25,822 slates'}
baseline (Step 6 LightGBM): val NDCG@10 0.4152 | 9.8 min
+ purchase sequence: val NDCG@10 0.4136 | 6.3 min


,val NDCG@10,val HR@10,val Purchase-HR@10,trees,gain share of new features,min
baseline (Step 6 LightGBM),0.4152,0.7162,0.6509,895.0,0.0000,9.7830
+ purchase sequence,0.4136,0.7155,0.6473,503.0,0.0011,6.2745


In [9]:
CHOSEN = GRID['val NDCG@10'].idxmax()
print(f'chosen on validation: {CHOSEN}')
pd.concat({n: IMP[n].drop(BASE) for n in VARIANTS if n != 'baseline (Step 6 LightGBM)'}, axis=1).round(4)

chosen on validation: baseline (Step 6 LightGBM)


,+ purchase sequence
seq_score,0.0003
seq_support_log,0.0004
seq_source_age_days,0.0005


## 6. Test results with paired bootstrap

Positive difference = first model better. The bootstrap resamples whole test slates (2,000 draws); `p` is one-sided.

In [10]:
M = {n: slate_metrics(U['test'], Y['test'], SCORES[n]['test']) for n in VARIANTS}

def paired(ma, mb, metric, n_boot=2000, seed=SEED):
    if metric == 'NDCG@10':
        d = ma['ndcg'] - mb['ndcg']
    else:
        d = ma['buy_hit'][ma['has_buy']].astype(float) - mb['buy_hit'][mb['has_buy']].astype(float)
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0, len(d), (200, len(d)))].mean(1) for _ in range(n_boot // 200)])
    return {'difference': d.mean(), 'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
            'p (one-sided)': max((boot <= 0).mean(), 1 / n_boot)}

TEST = pd.DataFrame({n: summary(M[n]) for n in VARIANTS}).T
BASE_NAME = 'baseline (Step 6 LightGBM)'
TESTS = {f'{n} vs baseline [{k}]': paired(M[n], M[BASE_NAME], k)
         for n in VARIANTS if n != BASE_NAME for k in ('NDCG@10', 'Purchase-HR@10')}
print(TEST.round(4).to_string()); print()
pd.DataFrame(TESTS).T.round(5)

                            NDCG@10   HR@10  Purchase-HR@10
baseline (Step 6 LightGBM)   0.4160  0.7194          0.6659
+ purchase sequence          0.4153  0.7191          0.6682



,difference,ci_low,ci_high,p (one-sided)
+ purchase sequence vs baseline [NDCG@10],-0.00070,-0.00166,0.00026,0.924
+ purchase sequence vs baseline [Purchase-HR@10],0.00227,-0.00324,0.00778,0.237


### Where the new features help

Segments where the new signals can act: records with an order in the last 30 days (the only ones a purchase-sequence score can exist for), slates containing an item the user clicked in the last 7 days (where the click split matters), and slates where no interacted item has any popularity signal (close to Step 7's all-cold failure segment).

In [11]:
te = feat.loc[part['test']]
sl = pd.DataFrame({'user': M[BASE_NAME]['user']}).set_index('user')
sl['recent order (30 d)'] = te.groupby('user')['seq_source_age_days'].apply(lambda x: x.notna().any())
sl['clicked a candidate (7 d)'] = te.groupby('user')['uc_click_7d'].max() > 0
pop = te[['i_users_click_7d', 'i_users_cart_90d', 'i_users_ord_365d']].fillna(0).sum(axis=1)
sl['no interacted item has popularity'] = pop[te['label'] > 0].groupby(te.loc[te['label'] > 0, 'user']).max().eq(0)
seg = []
for col in ['recent order (30 d)', 'clicked a candidate (7 d)', 'no interacted item has popularity']:
    for val in [True, False]:
        mask = sl[col].to_numpy() == val
        row = {'segment': col, 'value': val, 'slates': int(mask.sum())}
        for n in [BASE_NAME, CHOSEN]:
            row[n] = M[n]['ndcg'][mask].mean()
        row['difference'] = row[CHOSEN] - row[BASE_NAME] if CHOSEN != BASE_NAME else 0.0
        seg.append(row)
SEG = pd.DataFrame(seg)
SEG.round(4)

,segment,value,slates,baseline (Step 6 LightGBM),difference
0,recent order (30 d),True,13941,0.4119,0.0
1,recent order (30 d),False,11881,0.4208,0.0
2,clicked a candidate (7 d),True,5402,0.4970,0.0
3,clicked a candidate (7 d),False,20420,0.3946,0.0
4,no interacted item has popularity,True,4448,0.1801,0.0
5,no interacted item has popularity,False,21374,0.4651,0.0


## 7. Refresh the final ensemble

Step 6's final model is a within-slate z-score blend with weight 0.5 on the neural ranker (chosen on validation in Step 6). Here the neural test scores saved by Step 6 are reused unchanged, and only the LightGBM half is swapped for the chosen variant. Keeping the Step 6 weight avoids choosing anything on test. If the chosen variant is the baseline, this section reproduces the Step 6 ensemble and serves as a consistency check.

In [12]:
def zscore(user, x):
    g = pd.Series(x).groupby(user)
    return ((x - g.transform('mean').to_numpy()) / (g.transform('std', ddof=0).to_numpy() + 1e-9))

ENS = None
if PRED_PATH.exists():
    old = pd.read_parquet(PRED_PATH, columns=['user', 'wid', 'neural', 'ensemble'])
    te_keys = pd.DataFrame({'user': U['test'], 'wid': feat['wid'].to_numpy()[part['test']]})
    j = te_keys.merge(old, on=['user', 'wid'], how='left', validate='one_to_one')
    assert j['neural'].notna().all(), 'Step 6 predictions do not cover this test set'
    ENS = {'Step 6 ensemble (published)': j['ensemble'].to_numpy(),
           'neural (Step 6)': j['neural'].to_numpy(),
           f'new ensemble: neural + {CHOSEN}': 0.5 * zscore(U['test'], j['neural'].to_numpy())
                                               + 0.5 * zscore(U['test'], SCORES[CHOSEN]['test'])}
    for n, s in ENS.items():
        M[n] = slate_metrics(U['test'], Y['test'], s)
    ens_names = list(ENS)
    ENS_TABLE = pd.DataFrame({n: summary(M[n]) for n in ens_names}).T
    for k in ('NDCG@10', 'Purchase-HR@10'):
        TESTS[f'new ensemble vs Step 6 ensemble [{k}]'] = paired(M[ens_names[2]], M[ens_names[0]], k)
        TESTS[f'new ensemble vs {CHOSEN} [{k}]'] = paired(M[ens_names[2]], M[CHOSEN], k)
    print(ENS_TABLE.round(4).to_string()); print()
    display(pd.DataFrame(TESTS).T.iloc[-4:].round(5))
else:
    print(f'{PRED_PATH} not found: run Step 6 first to compare against the final ensemble')

                                                   NDCG@10   HR@10  Purchase-HR@10
Step 6 ensemble (published)                         0.4201  0.7236          0.6756
neural (Step 6)                                     0.4185  0.7235          0.6750
new ensemble: neural + baseline (Step 6 LightGBM)   0.4201  0.7234          0.6753



,difference,ci_low,ci_high,p (one-sided)
new ensemble vs Step 6 ensemble [NDCG@10],-0.00002,-0.00019,0.00014,0.5600
new ensemble vs baseline (Step 6 LightGBM) [NDCG@10],0.00404,0.00268,0.00539,0.0005
new ensemble vs Step 6 ensemble [Purchase-HR@10],-0.00032,-0.00162,0.00065,0.8100
new ensemble vs baseline (Step 6 LightGBM) [Purchase-HR@10],0.00940,0.00259,0.01588,0.0045


In [13]:
def safe_write(path, write):
    tmp = path.with_name('tmp_' + path.name)
    write(tmp); os.replace(tmp, path)
    print('saved', path)

newf = pd.DataFrame({'user': cu, 'wid': ORIG_WID, **{c: NEW[c] for c in CLICK_FEATS + SEQ_FEATS}})   # features.parquet row order
res = pd.concat([GRID.assign(stage='LightGBM variants (val)'), TEST.assign(stage='LightGBM variants (test)'),
                 pd.DataFrame(TESTS).T.assign(stage='paired bootstrap (test)'),
                 SEG.set_index(SEG['segment'] + ' = ' + SEG['value'].astype(str)).drop(columns=['segment', 'value'])
                    .assign(stage='segments (test NDCG@10)')]
                + ([ENS_TABLE.assign(stage='ensemble (test)')] if ENS is not None else []))
res.index.name = 'model / comparison'
pred = pd.DataFrame({'user': U['test'], 'wid': feat['wid'].to_numpy()[part['test']], 'label': Y['test'],
                     'lgbm_new': SCORES[CHOSEN]['test']})
if ENS is not None:
    pred['ensemble_new'] = ENS[ens_names[2]]
safe_write(OUT_DIR / f'step10_new_features{SFX}.parquet', lambda p: newf.to_parquet(p, index=False))
safe_write(OUT_DIR / f'step10_results{SFX}.csv', lambda p: res.to_csv(p))
safe_write(OUT_DIR / f'step10_test_predictions{SFX}.parquet', lambda p: pred.to_parquet(p, index=False))
try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass

saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step10_new_features.parquet
saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step10_results.csv
saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step10_test_predictions.parquet
Drive flushed and unmounted - safe to close the runtime


## 8. Takeaways

**Result: purchase-sequence affinity is not adopted; the Step 6 ensemble stays the final model (test NDCG@10 0.4201).**

1. **The feature builds correctly at full scale.** 3.1 M eligible purchase episodes in the training records give 18,876 supported category pairs. The vectorised scores match the branch's own `transform()` exactly. 46% of candidates and 54% of slates receive a score, and in 46% of slates the score differs between candidates, so it *could* change rankings.
2. **It does not improve ranking.** On validation, NDCG@10 falls from 0.4152 to 0.4136, so the baseline is chosen. On test, the difference is −0.0007 NDCG@10 (95% CI −0.0017 to +0.0003) and +0.0023 Purchase-HR@10 (CI −0.0032 to +0.0078); neither is significant. The three columns take 0.1% of the model's split gain.
3. **Why a real pattern doesn't help here.** Candidates with a score are bought twice as often in pooled data (0.18% vs 0.09%), but that reflects *who* buys (recent purchasers buy more), not *which item* in the slate. In search, the query already fixes the category, so "people buy B after A" rarely separates candidates within one slate. Existing features already carry the user's category affinity (`ucat3_*`, `ucat2_*`, `slate_cat3_share`). This repeats Step 4d's finding for `cat3_cross_lift_max`.
4. **Click context is settled by Step 8** (no gain), so it was built and described here but not retrained.
5. **The ensemble refresh is a consistency check.** With the baseline chosen, neural + LightGBM gives 0.4201 vs the published 0.4201 (difference −0.00002, CI −0.0002 to +0.0001). This confirms Step 6 is reproducible from saved predictions. The ensemble still beats LightGBM alone by +0.0040 NDCG@10 (CI +0.0027 to +0.0054) and +0.0094 Purchase-HR@10.

**Where this leaves the model.** Every scalar feature tried since Step 4b has been null on the full model: seasonality (4c), click-source split (8) and purchase sequences (9). Step 7 showed that 62% of remaining misses have no usable signal in the data. The remaining headroom is a data limit, not a feature-engineering gap.